# The Bootstrap
## DS-3001: Foundations of Machine Learning

Content adapted from Terence Johnson (UVA)

### Loading Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # For changing directory

# To mount your google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_DS_3001_folder = '/content/drive/MyDrive/DS-3001/03_Tuning_Testing_Evaluating'
# path_to_DS_3001_folder = ''

# Update the path to your folder for the class
# Where you stored the data from the previous noteboook
os.chdir(path_to_DS_3001_folder)

## The Bootstrap

- **How can we get more data?**
  - In data science, we just have the data we have, we can't really get new samples without gathering more data, right? Or can we?
- **Introducing the Bootstrap:**
  - Today we're going to learn about a "data-driven central limit theorem" that we can use to estimate the sampling distribution of a statistic prediction purposes.

## Brief Review: The CDF and ECDF

### Example: The Probability Function of the Sum of Two Dice
- The sample space is:
$$
\left[ \begin{array}{cccccc}
(1\cdot,1\cdot) & (1\cdot,2\cdot) & (1\cdot,3\cdot) & (1\cdot,4\cdot) & (1\cdot,5\cdot) & (1\cdot,6\cdot) \\
(2\cdot,1\cdot) & (2\cdot,2\cdot) & (2\cdot,3\cdot) & (2\cdot,4\cdot) & (2\cdot,5\cdot) & (2\cdot,6\cdot) \\
(3\cdot,1\cdot) & (3\cdot,2\cdot) & (3\cdot,3\cdot) & (3\cdot,4\cdot) & (3\cdot,5\cdot) & (3\cdot,6\cdot) \\
(4\cdot,1\cdot) & (4\cdot,2\cdot) & (4\cdot,3\cdot) & (4\cdot,4\cdot) & (4\cdot,5\cdot) & (4\cdot,6\cdot) \\
(5\cdot,1\cdot) & (5\cdot,2\cdot) & (5\cdot,3\cdot) & (5\cdot,4\cdot) & (5\cdot,5\cdot) & (5\cdot,6\cdot) \\
(6\cdot,1\cdot) & (6\cdot,2\cdot) & (6\cdot,3\cdot) & (6\cdot,4\cdot) & (6\cdot,5\cdot) & (6\cdot,6\cdot)
\end{array} \right]
$$
- We're interested in the sum of the two die $(d_1,d_2)$ as our random variable, $R = d_1 + d2$, and the probability function and CDF, $F_R(x) = pr[d_1 + d_2 \le x]$

### The Cumulative Distribution Function

- We're intereseted in describing how a random variable $R$ is *distributed* over the values it takes.

- The *cumulative distribution function* or CDF is defined as
$$
F(x) = pr[R\le x]
$$

- **What does this mean?**
  - For any number $x$, this gives the probability that the random variable $R$ takes values below $x$.

- **What does the CDF look like?**
  - The function is increasing, since as $x$ goes up, the number of values $R$ for which $R\le x$ gets larger.
  - It is bounded below by 0 and above by 1, since it is a probability.

**Plotting the PDF of the sum of 2 die**

In [ ]:
# Possible values of a dice roll
D = [1,2,3,4,5,6]

# A list of probabilities to save
pr = np.zeros(11)

# The values that the sum can take on
R = np.arange(2,13,1)

# Sum over the possible values of dice 1
for i in range(6):
    # Sum over the possiblev alues of dice 2
    for j in range(6):
        pr[i+j] = pr[i+j] + 1/36 # Adding 1/36th of the probability to this combination

# Creating a plot of the PDF for this process
plt.scatter(R,pr)
plt.plot(R,pr)
plt.xlabel("d1+d2")
plt.ylabel("pr[d1+d2]")
plt.title('Probability Function of the Sum of Two Die')
plt.show()

# Creating a data frame and printing it out
df = pd.DataFrame({'sum': R,'pr': pr})
df

**The CDF of the sum of 2 die**

In [ ]:
# The CDF is just the cumulative sum of the probabilties
F = np.cumsum(pr)

# Plotting the CDF
plt.scatter(R,F)
plt.plot(R,F)
plt.xlabel("x")
plt.ylabel("F(x) = pr[d1+d2 <= x]")
plt.title('Distribution Function of the Sum of Two Die')
plt.show()

# Creating a data frame of the possible sums and the CDF
df = pd.DataFrame({'x':R,'F':F})
df


**What we've seen before: The PDF and CDF of the Noraml Distribution**

In [ ]:
from scipy.stats import norm

# Values of
grid = np.linspace(-3,3,30)
pdf = norm.pdf(grid)
cdf = norm.cdf(grid)

# Plotting the PDF
plt.scatter(grid,pdf)
plt.plot(grid,pdf)
plt.xlabel("x")
plt.ylabel("pdf(x)")
plt.title('Standard Normal Probability Density Function (PDF)')
plt.show()

# Plotting the CDF
plt.scatter(grid,cdf)
plt.plot(grid,cdf)
plt.xlabel("x")
plt.ylabel("cdf(x)")
plt.title('Standard Normal Probability Distribution Function (CDF)')
plt.show()


## The Empirical CDF
- Above, we used the formula for the normal distribution to calculate the PDF and CDF. But how do we model a distribution function $F$ ***using data***? Answer: The Empirical CDF.

- Define some indicator function: $L(x_i, x)=1$. If $x_i$ is **L**ess than $x$, then $L$ equals 1. If $x_i$ is greater than $x$, then $L$ equals 0. In notation:

$$
L(x_i, x) = \begin{cases} 1, & x_i \le x \\ 0, & x_i > x\end{cases}
$$

- Defining the *empirical cumulative distribution function* (ECDF):
$$
F_n(x) = \dfrac{1}{n} \sum_{i=1}^n L(x_i,x)
$$

- This is just a sample proportion for each $x$. This is a cousin of the histogram (representing the distribution function rather than the density function)


## Convergence of the ECDF
- **Holding $x$ fixed, the law of large number (LLN) implies that the ECDF $F_n(x)$ converges in probability to the true CDF $F(x)$ as our sample size $n$ gets large**

- **We can use the ECDF and the LLN to estimate distributions of random variables**:
  - It's not just some statistic like the mean or variance, it's estimating the whole distribution of the random variable.

**Using the code below, test different values of N to sample and see how it changes how close the ECDF is to the CDF:**

In [ ]:
## ECDF, Dice Example, 50 rolls

# Values for the possible values of the dice face
R = np.arange(2,13,1)

# Possible dice faces we could have
faces = np.array([1,2,3,4,5,6])

# Randomly draw the values of both dice faces
N = 50
d1 = np.random.choice(faces, size=N)
d2 = np.random.choice(faces, size=N)

# Calculate the roll
roll = d1+d2

# Define an array for the values of the ECDF function
ecdf = np.zeros(11)

# Loop over the possible values that the sum of the roll can be
for i in range(11):
    # Calculate the ECDF
    ecdf[i] = (1/N)*np.sum(
        roll <= R[i] # Calculating the L(x_i, x)
    )

# Plot the ECDF
plt.scatter(R,ecdf,label='ECDF')
plt.plot(R,ecdf,label='ECDF')

# Plot the true CDF
plt.scatter(R,F,label='CDF')
plt.plot(R,F,label='CDF')

# Add Labels
plt.xlabel("x")
plt.ylabel("Freq")
plt.legend(loc='lower right')
plt.title(f'Empirical and Theoretical CDF\nN = {N}')
plt.show()


# Bootstrapping

## Simulating Uncertainty
- **What is a statistic?**
  - A **statistic** $S$ is any function of the sample data
  - Examples:
    - The mean of the data: $m(x)$
    - The value of the regression coefficients: $\hat{b}(x)$
    - The predictions of a model: $\hat{y}(x)$

- **Some assumptions:**
  - We assume the data are independent and identically distributed (**iid**):
    - **Independent**: Each observation $x_i$ is determined independently of the other observations $x_j$, $j\neq i$
    - **Identically distributed:** All observations come from the same sampling distribution.

- **Measuring the uncertainty of a statistic:**
  - Suppose we've estimated a statistic, $S$, and want to know how uncertain we should be about its value.
  - Examples:
    - The variance of the predictions
    - The value of a regression coefficient
    - The number of clusters that $k$-MC picks

- **The central question:**
  - ***How can we "mimic" the arrival of new data and provide estimates about the uncertainty in our models?***

## The Bootstrap

- **Defining the Bootstrap process:**
  - We can **bootstrap** the distribution of statistic $S$ as follows:

    1. Pick a large number, $R$, to be the number of **bootstrap replications**

    2. Select $n$ observations at random from your data, **with replacement**, and call this sample $t$,  $X_t$

    3. Compute the statistic of interest $S_t = S(X_t)$ for this sample, and store its value

    4. Repeat steps 2--3  for $t=1,2,...,R$, and then compute the ECDF for the stored values $(S_1, S_2, ..., S_R)$

- **Why does this work?**
  - As $R$ and $n$ get large, our bootstrapped ECDF $F_{n,R}(S)$ of statistic $S$ will converge to the true distribution $F(S)$

## **What is happening?**
- **We can't create new data, but we can mimic the process of creating new data:**

  - We typically can't create new data, and this is inconvenient. We have what we have.

  - ***If the data are iid***, however, we can *mimic* the process of collecting data by resampling.

  - We close our eyes and draw a new sample at random *with replacement* from the existing data, equal to the size of the original data set.

- **Each bootstrapped sample, $X_t$ is a new sample from the same data generation process:**

  - If you have $N$ observations, there are $N^N$ ways of doing this: That is an astronomically large number (e.g. $10^{10} = 10,000,000,000$)

  - This is effectively a new sample from the same data generating process: It will have slightly different properties and values, and provide slightly different estimates

  - We can repeat this process many times, simulating a situation where we *could* create as many datasets as we wanted that have similar properties to the original

- **When does it it work:**
  - Because of the LLN, the statistics we compute from each bootstrap dataset create an ECDF of $(S_1,...,S_R)$ that gets close to the true distribution of $S$ if:
    1. The data are really iid
    2. The number of draws $R$ is sufficiently large
    3. We have a sufficiently large dataset in $n$

## The Sampling Distribution

- **What is the sampling distribution:**
  - In statistics, the **sampling distribution** is the distribution of the thought experiment:
  - *"What if we ran this experiment thousands of times, and computed the statistic of interest each time. How would that quantity be distributed?"*

- **The bootstrap simulates the sampling distribution:**
  - What we are doing with the bootstrap is using our data to **simulate the sampling distribution** and try to make inferences about it and the true parameter value

# Example: Comparing the True Sampling Distribution versus Bootstrapping

## Simulation and Repetition
- I'm alleging that, if the observations are iid, the results from these two processes are very similar:

    - Run an experiment $T$ times with sample size $N$
    - Run an experiment with sample size $N$ once, then resample it $T$ times with replacement

- Let's look at a simulation

**Try different values of N, T, and R. How does the approximation change?**

In [ ]:
# Run an experiment with N observations T times
N = 5000
T = 10000

# The true distribution will have mean 0 and standard deviation 1
mu = 0
sigma = 1

### Run an experiment T times with sample size N
# A list to save the values of the mean
mean = []
for t in range(T):
    # Draw a sample from the normal distribution with size N, T times
    sample = np.random.normal(mu,sigma,size=N)

    # Save the mean of the sample
    mean.append( np.mean(sample) )

### Run an experiment with N observations once, resample R times
R = 10000 # Number of times to resample

# Gather one sample
sample = np.random.normal(mu,sigma,size=N)

# List to save the mean
mean_bs = []
for t in range(R):
    # Resample from the single sample that we have with replacement
    sample_t = np.random.choice(
        sample, # The sample that we're drawing from, our data
        size=N, # The number of observations in our bootstrapped sample
        replace=True # With replacement, otherwise we just have the same data set
    )

    # Save the mean
    mean_bs.append( np.mean(sample_t) )


In [ ]:
# Plot the ECDF of the experiment T times with sample size N
sns.ecdfplot(mean, label = 'N observations T times')

# Plot the ECDF of the experiment with N observations once, resample R times
sns.ecdfplot(mean_bs, label = 'Bootstrapped')
plt.title(f'ECDF of the Mean\nN = {N}; T = {T}; R = {R}')
plt.xlabel('The Sample Mean')
plt.legend()
plt.show()

# Plot the PDF of the experiment T times with sample size N
sns.kdeplot(mean, label = 'N observations T times')

# Plot the PDF of the experiment with N observations once, resample R times
sns.kdeplot(mean_bs, label = 'Bootstrapped')
plt.title(f'PDF of the Mean\nN = {N}; T = {T}; R = {R}')
plt.xlabel('The Sample Mean')
plt.legend(bbox_to_anchor = (1.5, 0.6))
plt.show()

# Confidence Intervals

## Communicating Uncertainty: Confidence Intervals

- When we have a bootstrapped distribution, it is nice to convert the picture into numbers or some summary statement.

- **Defining the confidence interval:**
  - People often report an *$1 - \alpha$% confidence interval*.

  - The confidence interval is created with a lower limit of $(1-\alpha)/2$-quantile and an upper limit $[1-(1-\alpha)/2]$-quantile.

  - The value of $\alpha$ is usually:
    - $\alpha$ = 90%, lower bound is the 5th quantile, upper bound is the 95th quantile.
    - $\alpha$ = 95%, lower bound is the 2.5th quantile, upper bound is the 97.5th quantile.
    - $\alpha$ = 99%, lower bound is the 0.5th qunatile, upper bound is the 99.5th quantile.

- **Definitions:**
  - **Confidence level:** The value $\alpha$
  - **Significance level:** The value $1 - \alpha$

- **Understanding the confidence interval in words:**
  - If we repeatedly draw samples (assuming each sample is iid and we use a sufficient sample size), and compute a confidence interval for each sample using this bootstrapping method, then approximately $\alpha$% of those intervals will contain the true population parameter.
  - In other words: **Imagine you did this experiment many times. For $\alpha$% of the confidence intervals created by those trials, the true value of the parameter would be inside the confidence interval.**
  - This is **NOT** the same thing as saying "With probability $\alpha$, the value being estimated by $S$ is in this interval."
  - The parameter being estimated by $S$ is not random, so statements about the probability it is in one range or another don't quite make sense (it is or is not in a given range).

- This idea can be extended to hypothesis testing for significance: If values like 0 are outside the confidence interval, such a value seems inconsistent with the data we have available.

**Plotting confidence interval for a single sample:**
  - We'll look at a normal distribution, pulling one sample and bootstrapping it.
  - We'll plot the PDF and confidence intervals for this example.

In [ ]:
# Mean and standard deviation of the population distribution
mu = 10
std = 1
N = 1000

# Number of times to bootstrap
R = 10000

# Draw a single sample from the distribution
# This is like our single data set that we have to work with
np.random.seed(90)
sample = np.random.normal(mu, std, size=N)

# List to store the bootstrapped means
bootstrapped_means = []

# Loop for bootstrapping the sample
for t in range(R):

  # Bootstrap the sampled data set
  sample_t = np.random.choice(
      sample, # The sample that we're drawing from, our data
      size=N, # The number of observations in our bootstrapped sample
      replace=True # With replacement, otherwise we just have the same data set
  )

  # Save the mean
  bootstrapped_means.append( np.mean(sample_t) )

# Calculate the intervals for the 90, 95, 99 confidence intervals
L90 = np.quantile(bootstrapped_means, (1 - .90) / 2)
U90 = np.quantile(bootstrapped_means, 1 - (1 - .90) / 2)

L95 = np.quantile(bootstrapped_means, (1 - .95) / 2)
U95 = np.quantile(bootstrapped_means, 1 - (1 - .95) / 2)

L99 = np.quantile(bootstrapped_means, (1 - .99) / 2)
U99 = np.quantile(bootstrapped_means, 1 - (1 - .99) / 2)

# Plot the PDF of our bootstrapped means
sns.kdeplot(bootstrapped_means)
plt.title('PDF of the Mean with Confidence Intervals')
plt.xlabel('Bootstrapped Mean')

# Add the point estimate for the mean
plt.axvline(x = np.mean(bootstrapped_means), linestyle='dashed', color='black', label = 'Mean of Bootstrapped Means')

# Add the confidence lines
plt.axvline(x = L90, linestyle='dashed',color='red', label = '90% Confidence Interval')
plt.axvline(x = U90, linestyle='dashed',color='red')

plt.axvline(x = L95, linestyle='dashed',color='orange', label = '95% Confidence Interval')
plt.axvline(x = U95, linestyle='dashed',color='orange')

plt.axvline(x = L99, linestyle='dashed',color='green', label = '99% Confidence Interval')
plt.axvline(x = U99, linestyle='dashed',color='green')

plt.legend(bbox_to_anchor = (1.6, 0.6))
plt.show()


**How does our method work if we run this experiment many times?:**
  - Let's repeat this example with the normal distribution but this time do the experiment many times.
  - We'll sample from the normal distribution 100 times, bootstrap to estimate the confidence interval, and then check whether our estimated confidence interval either does or does not capture the true mean.
  - We'll have 100 confidence intervals, and we can see how many times this estimated confidence interval captured the true statistic.

In [ ]:
# Let's look at an example of estimating the mean from the normal distribution
from tqdm import tqdm

# Mean and standard deviation of the population distribution
mu = 10
std = 1
N = 1000

# The value of alpha to work with
alpha = 0.95

# The number of datasets to pull and then the number of times to bootstrap each
num_datasets = 100
R = 10000

center_means = []
lower_bounds = []
upper_bounds = []

# Set a seed for reproducibility
np.random.seed(90)

# Loop over the number of data sets to pull from
for i in tqdm(range(num_datasets)):
    # Draw a sample from the normal distribution
    sample = np.random.normal(mu, std, size=N)

    bootstrapped_means = []

    # Compute the mean of the sample using bootstrapping
    for t in range(R):
      # Bootstrap the sampled data set
      sample_t = np.random.choice(
          sample, # The sample that we're drawing from, our data
          size=N, # The number of observations in our bootstrapped sample
          replace=True # With replacement, otherwise we just have the same data set
      )

      # Save the mean
      bootstrapped_means.append( np.mean(sample_t) )

    # Calculate the lower and upper bounds
    bootstrapped_means = np.array(bootstrapped_means)
    lower_bound = np.quantile(bootstrapped_means, (1 - alpha) / 2)
    upper_bound = np.quantile(bootstrapped_means, (1 - ((1 - alpha) / 2)))

    # Append the results to our outer lists
    center_means.append(np.mean(bootstrapped_means))
    lower_bounds.append(lower_bound)
    upper_bounds.append(upper_bound)

# Plot the result
fig, ax = plt.subplots(figsize=(8, 8))

contains_mu_tracker = []

for i in range(num_datasets):
    # Check if the confidence interval contains the true value of the mean
    contains_mu = lower_bounds[i] <= mu <= upper_bounds[i]
    contains_mu_tracker.append(contains_mu)

    # Color based on if it captured or not
    color = "steelblue" if contains_mu else "tomato"

    # Plot the data point
    ax.plot([lower_bounds[i], upper_bounds[i]], [i, i], color=color, lw=2)
    ax.plot(center_means[i], i, "o", color=color)

# Get the number of intervals that contain the true value of mu
num_contains_mu = np.sum(contains_mu_tracker) / num_datasets
print(f'Based on this experiment, the interval captured the true mean {num_contains_mu}% of the time')

# Set a line for the true value of mu
ax.axvline(
    mu, color="black",
    linestyle="--", label=fr"True mean ($\mu$={mu})")
ax.set_xlabel("The mean of the Mean")
ax.set_ylabel("Sample index")
ax.set_title(
    f"Bootstrapped {int(alpha*100)}% Confidence Intervals\n"
    r"(blue = contains $\mu$, red = does not contain $\mu$)"
)
ax.legend()
plt.tight_layout()
plt.show()

**Same thing, this time with $\alpha=0.5$:**

In [ ]:
# Let's look at an example of estimating the mean from the normal distribution

# Mean and standard deviation of the population distribution
mu = 10
std = 1
N = 1000

# The value of alpha to work with
alpha = 0.5

# The
num_datasets = 100
R = 10000

center_means = []
lower_bounds = []
upper_bounds = []

# Set a seed for reproducibility
np.random.seed(90)

# Loop over the number of data sets to pull from
for i in tqdm(range(num_datasets)):
    # Draw a sample from the normal distribution
    sample = np.random.normal(mu, std, size=N)

    bootstrapped_means = []

    # Compute the mean of the sample using bootstrapping
    for t in range(R):
      # Bootstrap the sampled data set
      sample_t = np.random.choice(
          sample, # The sample that we're drawing from, our data
          size=N, # The number of observations in our bootstrapped sample
          replace=True # With replacement, otherwise we just have the same data set
      )

      # Save the mean
      bootstrapped_means.append( np.mean(sample_t) )

    # Calculate the lower and upper bounds
    bootstrapped_means = np.array(bootstrapped_means)
    lower_bound = np.quantile(bootstrapped_means, (1 - alpha) / 2)
    upper_bound = np.quantile(bootstrapped_means, (1 - ((1 - alpha) / 2)))

    # Append the results to our outer lists
    center_means.append(np.mean(bootstrapped_means))
    lower_bounds.append(lower_bound)
    upper_bounds.append(upper_bound)

# Plot the result
fig, ax = plt.subplots(figsize=(8, 8))

contains_mu_tracker = []

for i in range(num_datasets):
    # Check if the confidence interval contains the true value of the mean
    contains_mu = lower_bounds[i] <= mu <= upper_bounds[i]
    contains_mu_tracker.append(contains_mu)

    # Color based on if it captured or not
    color = "steelblue" if contains_mu else "tomato"

    # Plot the data point
    ax.plot([lower_bounds[i], upper_bounds[i]], [i, i], color=color, lw=2)
    ax.plot(center_means[i], i, "o", color=color)

# Get the number of intervals that contain the true value of mu
num_contains_mu = np.sum(contains_mu_tracker) / num_datasets
print(f'Based on this experiment, the interval captured the true mean {num_contains_mu}% of the time')

# Set a line for the true value of mu
ax.axvline(
    mu, color="black",
    linestyle="--", label=fr"True mean ($\mu$={mu})")
ax.set_xlabel("The mean of the Mean")
ax.set_ylabel("Sample index")
ax.set_title(
    f"Bootstrapped {int(alpha*100)}% Confidence Intervals\n"
    r"(blue = contains $\mu$, red = does not contain $\mu$)"
)
ax.legend()
plt.tight_layout()
plt.show()

# Applying the Bootstrap to a Model

## Estimating the Uncertainty in Model Parameters
- **Problem:**
  - We have our data $(y,X)$ and a model $y = b\cdot X$.
  - We want to quantify the uncertainty about the estimated coefficients in the model, say $\hat{b}$.
- **Solution:**
  - We can resample the data $(y,X)$ and run the regression on each bootstrap sample.
  - This creates a sequence of estimated coefficients. $(\hat{b}^1, \hat{b}^2, ... , \hat{b}^S)$.
  - We can see how "noisy" the coefficient $\hat{b}$ is and construct confidence intervals.

For this example, we can use the cardiac patient data that we've seen previously.

In [ ]:
# Load in the data set
cardiac_df = pd.read_csv('./data/CardiacPatientData.csv')
print('Shape of the data:', cardiac_df.shape)
cardiac_df.head()

# Drop some variables that have lots of NaN values
cardiac_df = cardiac_df.drop(
    [
        'Na','K', 'Cl', 'Urea','Ceratinine',
        'Alcoholic','Smoke','FHCD', 'TriageScore'
    ],
    axis=1
)

In [ ]:
# We can fit a logistic regression model to our data
from sklearn.linear_model import LogisticRegression

# Isolate the outcome and our design matrix
y = cardiac_df['Outcome']
X = cardiac_df.drop(['ID','Outcome'], axis=1)

# MinMax scale our features
def maxmin(z):
    z = (z-np.min(z))/(np.max(z)-np.min(z))
    return z

X = X.apply(maxmin)


# Set the model instance and fit
LR = LogisticRegression(penalty=None)
LR = LR.fit(X,y)

# Predict the probabilities
p_hat = LR.predict_proba(X)
p_hat = p_hat[:, 1]

# Creat a dataframe of the features and the coefficient
pd.DataFrame(
    {
        'variable':LR.feature_names_in_,
        'coefficient':LR.coef_[0]
    }
)

**Now, we'll boostrap our data and fit the model for each bootstrap. We can see how the coefficients change as a result:**

In [ ]:
# Bootstrapping the data
R = 1000 # Number of boostraps

# List to save the coefficients for each sample
bootstrapped_coefs = []
bootstrapped_predictions = []

# Set a seed for reproducibility
np.random.seed(90)

# Loop over for the number of bootstraps
for t in tqdm(range(R)):

    # Sample our data to get a bootstrap sample
    cardiac_df_t = cardiac_df.sample(
        cardiac_df.shape[0], # Want N samples for each bootstrapped sample
        axis=0, # Sampling the rows
        replace=True # Using replacement
    )

    ## Repeat the analysis with the bootstrap replication:

    # Isolate outcome and the design matrix
    y_t = cardiac_df_t['Outcome']
    X_t = cardiac_df_t.drop(['ID','Outcome'], axis=1)

    # Scale the design matrix
    X_t = X_t.apply(maxmin)

    # Create logisitc regression model and fit
    LR_t = LogisticRegression(penalty=None)
    LR_t = LR_t.fit(X_t,y_t)

    # Get predictions using the bootstrapped model on our original data
    p_hat = LR_t.predict_proba(X)
    p_hat = p_hat[:, 1]

    ## Save the results:
    bootstrapped_coefs.append(LR_t.coef_[0])
    bootstrapped_predictions.append(p_hat)

# Create a matrix from the bootstrapped coefficients
bootstrapped_coefs = np.stack(bootstrapped_coefs)
bootstrapped_predictions = np.stack(bootstrapped_predictions)

**Plotting the distribution of coefficients according to bootstrap:**

In [ ]:
# Analyze the distribution of the regression coefficients

# Define an alpha
alpha = 0.95

# Loop over each feature
for k in range(len(LR.feature_names_in_)):
    # Print what the variable is and our original estimate
    print('Variable: ', LR.feature_names_in_[k])
    print('Estimate: ', LR.coef_[0][k])

    # Pull out the boostrapped estimates of the coefficient
    values = bootstrapped_coefs[:, k]

    # Compute the confidence interval for the coefficient
    L = np.round( np.quantile(values, (1 - alpha) / 2 ), 3 )
    U = np.round( np.quantile(values, 1 - (1 - alpha) / 2), 3 )
    print('95% Confidence Interval: (',L,',',U,')')

    # Create ECDF plots of the estimates
    ax = sns.ecdfplot(values)

    # Plot the vertical lines of the point estimate, lower / upper bound
    ax.axvline(x = LR.coef_[0][k], linestyle='dashed', color='black') # Point estimate
    ax.axvline(x = L, linestyle='dashed', color='red') # Lower bound
    ax.axvline(x = U, linestyle='dashed', color='red') # Upper bound
    ax.set_title(f'ECDF of Coefficient for {LR.feature_names_in_[k]}\n{np.round(alpha*100)}% Confidence Interval')

    plt.show()

    # Plot the PDF as well
    ax = sns.kdeplot(values)

    # Plot the vertical lines of the point estimate, lower / upper bound
    ax.axvline(x = LR.coef_[0][k], linestyle='dashed', color='black') # Point estimate
    ax.axvline(x = L, linestyle='dashed', color='red') # Lower bound
    ax.axvline(x = U, linestyle='dashed', color='red') # Upper bound
    ax.set_title(f'PDF of Coefficient for {LR.feature_names_in_[k]}\n{np.round(alpha*100)}% Confidence Interval')

    plt.show()

    print('\n \n \n')


**Plotting the distribution of the predictions for individual observations in our model:**

In [ ]:
# Isolate 20 observations to plot

# Set a seed for reproducibility
np.random.seed(180)

# Pick out 20 random samples to plot the distribution of predicted values
n = bootstrapped_predictions.shape[1]
observation_to_show = np.random.choice(n, 20)

subsample_predictions = bootstrapped_predictions[:, observation_to_show]

# Define our value of alpha
alpha = 0.95

point_estimate = np.mean(subsample_predictions, axis = 0)
lower = np.quantile(subsample_predictions, (1 - alpha) / 2, axis = 0)
upper = np.quantile(subsample_predictions, 1 - (1 - alpha) / 2, axis = 0)

# Plot the results
observation_index = np.arange(len(point_estimate))

# Plot the confidence interval band
plt.errorbar(
    observation_index, # The x-axis is based on what observation we're looking at
    point_estimate, # The dot will be our point estimate
    yerr=[point_estimate - lower, upper - point_estimate], # The bars will be based on our lower and upper limits
    fmt='o', # We want a
    color='steelblue',
    ecolor='steelblue',
    capsize=3,
    label=f'Point Estimate +- {alpha * 100}% CI',
    alpha = 0.5
)

# We can plot the true outcomes
plt.scatter(
    observation_index,
    cardiac_df['Outcome'].iloc[observation_to_show],
    color='black', s=30,
    label='True outcome'
)

# Add layers
plt.xlabel('Observation Index')
plt.ylabel('Predicted Probability')
plt.title('Bootstraped Confidence Interval for Predicted Probability by Observation')
plt.legend(bbox_to_anchor = (1.01, 0.6))
plt.show()

## Conclusion
- We're just scratching the absolute surface of bootstrapping and resampling methods
- You can use the bootstrap in any situation where you think: "The data could have been slightly different, and my estimates/predictions would have changed as a result. How much uncertainty should I realistically have?"
- The focus of the class is not on **statistical inference**, which is what the bootstrap is for, but instead on **prediction**
- The key insight of bootstrapping is that you can use resampling to create many "hypothetical" data sets that can be used to better understand the variation in the data and improve model performance